# Stream Ouro on one GPU

Use the **Python (loop-steer)** kernel. Run the setup cells in order, then rerun the prompt cell for experiments. The model and tokenizer stay available for activation steering. GPU 0 is the default; change it below before importing PyTorch and restart the kernel when switching GPUs.

In [ ]:
import os
import sys
from pathlib import Path

GPU_ID = "0"  # Physical GPU index: "0" or "1".
TOTAL_UT_STEPS = 4

if "torch" in sys.modules and os.environ.get("CUDA_VISIBLE_DEVICES") != GPU_ID:
    raise RuntimeError("Restart the kernel before changing GPU_ID.")

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if Path(sys.prefix).resolve() != (ROOT / ".venv").resolve():
    raise RuntimeError("Choose the Python (loop-steer) kernel from this project's .venv.")

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID


In [ ]:
import torch
import transformers
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from loop_steer import MODEL_ID, MODEL_REVISION
from loop_steer.streaming import stream_chat

assert torch.cuda.is_available(), "A working NVIDIA GPU is required."
assert torch.cuda.device_count() == 1, "The kernel should see exactly one GPU."
print(f"Physical GPU {GPU_ID} → cuda:0: {torch.cuda.get_device_name(0)}")
print(f"PyTorch {torch.__version__}, CUDA {torch.version.cuda}, Transformers {transformers.__version__}")

In [ ]:
# Reuse the configured Hugging Face cache; uncached weights are about 2.8 GB.
# The pinned revision contains the upstream Ouro KV cache fix.
config = AutoConfig.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True,
)
config.total_ut_steps = TOTAL_UT_STEPS
config.early_exit_threshold = 1.0

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, code_revision=MODEL_REVISION,
    trust_remote_code=True, config=config,
    dtype=torch.bfloat16, device_map="cuda:0",
    attn_implementation="sdpa",
).eval()
print(f"Loaded {len(model.model.layers)} shared layers × {config.total_ut_steps} recurrent passes")

In [ ]:
messages = [{"role": "user", "content": "Solve: If 2x + 3 = 11, what is x? Explain the steps."}]

# Change this prompt and rerun this cell. Output arrives as text is generated.
response = ""
for text in stream_chat(model, tokenizer, messages, max_new_tokens=256, temperature=1.0, top_p=0.7):
    print(text, end="", flush=True)
    response += text
print()

## Activation hooks

Attach PyTorch hooks to `model.model.layers[layer_index]` before running the prompt cell. Each shared layer executes once per recurrent pass: normally **four hook calls per model forward**. Use `register_forward_hook(..., with_kwargs=True)` and the `current_ut` keyword (zero-based) to steer a specific recurrence. Decoder outputs are tensors in this pinned model implementation. Hooks run in the generation thread; remove their handles after generation finishes. `model(..., output_hidden_states=True)` is also available for collecting activations.

## Optional: stream through vLLM

In another terminal, run `uv run python scripts/serve_vllm.py --gpu 1` and wait for the server to finish starting. Then set `RUN_VLLM = True` below. This server owns its own model; notebook hooks affect the Transformers model. vLLM always runs all recurrent steps.

In [ ]:
RUN_VLLM = False

if RUN_VLLM:
    from openai import OpenAI

    with OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="local", timeout=300.0) as client:
        with client.chat.completions.create(
            model=MODEL_ID, messages=messages, stream=True,
            max_tokens=256, temperature=1.0, top_p=0.7,
        ) as stream:
            for chunk in stream:
                if chunk.choices:
                    print(chunk.choices[0].delta.content or "", end="", flush=True)
        print()

## Release GPU memory when finished

Set `RELEASE_MODEL = True` to unload the Transformers model before using its GPU for another process. Reload the model cell to continue. Restarting the kernel also releases GPU memory and any experiment references.

In [ ]:
RELEASE_MODEL = False

if RELEASE_MODEL:
    import gc

    del model
    gc.collect()
    torch.cuda.empty_cache()
    print("Released the notebook model.")